# Single-Cell RNA-seq QC and Visualization

This notebook performs standard QC and visualization for scRNA-seq data:
- QC metrics (genes, counts, mito%)
- UMAP visualization
- Marker gene expression

**Input**: AnnData file (.h5ad)
**Outputs**: QC plots, UMAP, marker plots

In [ ]:
# Parameters (can be overridden by papermill)
MIN_GENES = 200
MAX_GENES = 5000
MAX_MITO_PCT = 20
N_TOP_GENES = 2000
MARKER_GENES = ['CD3D', 'CD14', 'CD79A', 'NKG7', 'PPBP']  # Default markers

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from galaxy_notebook import get, put, list_inputs

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, frameon=False)

In [ ]:
# List available inputs
print("Available inputs:")
for inp in list_inputs():
    print(f"  [{inp['index']}] {inp['name']}")

In [ ]:
# Load AnnData
input_path = get(0, format='path')
adata = sc.read_h5ad(input_path)
print(f"Loaded: {adata.n_obs} cells x {adata.n_vars} genes")
adata

## 1. QC Metrics

In [ ]:
# Calculate QC metrics
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, inplace=True)

# Plot QC metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sc.pl.violin(adata, 'n_genes_by_counts', ax=axes[0], show=False)
axes[0].set_title('Genes per cell')

sc.pl.violin(adata, 'total_counts', ax=axes[1], show=False)
axes[1].set_title('Total counts per cell')

sc.pl.violin(adata, 'pct_counts_mt', ax=axes[2], show=False)
axes[2].set_title('Mitochondrial %')

plt.tight_layout()
put(fig, 'qc_metrics.png')
plt.show()

In [ ]:
# QC scatter plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', ax=axes[0], show=False)
sc.pl.scatter(adata, x='total_counts', y='pct_counts_mt', ax=axes[1], show=False)

plt.tight_layout()
put(fig, 'qc_scatter.png')
plt.show()

## 2. UMAP Visualization

In [ ]:
# Check if UMAP already computed
if 'X_umap' not in adata.obsm:
    print("Computing UMAP...")
    # Basic preprocessing if needed
    if 'log1p' not in adata.uns:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
    
    sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES)
    sc.tl.pca(adata)
    sc.pp.neighbors(adata)
    sc.tl.umap(adata)
    
    # Clustering if not present
    if 'leiden' not in adata.obs:
        sc.tl.leiden(adata)

print("UMAP coordinates available")

In [ ]:
# Plot UMAP
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# By cluster
cluster_key = 'leiden' if 'leiden' in adata.obs else ('louvain' if 'louvain' in adata.obs else None)
if cluster_key:
    sc.pl.umap(adata, color=cluster_key, ax=axes[0], show=False, legend_loc='on data')
    axes[0].set_title(f'UMAP - {cluster_key} clusters')
else:
    sc.pl.umap(adata, ax=axes[0], show=False)
    axes[0].set_title('UMAP')

# By QC metric
sc.pl.umap(adata, color='n_genes_by_counts', ax=axes[1], show=False)
axes[1].set_title('UMAP - Genes per cell')

plt.tight_layout()
put(fig, 'umap_clusters.png')
plt.show()

## 3. Marker Gene Expression

In [ ]:
# Filter to markers that exist in the data
available_markers = [g for g in MARKER_GENES if g in adata.var_names]
print(f"Available markers: {available_markers}")

if available_markers:
    fig = sc.pl.umap(adata, color=available_markers, ncols=3, return_fig=True, show=False)
    put(fig, 'marker_expression.png')
    plt.show()
else:
    print("No marker genes found in dataset")

## 4. Summary

In [ ]:
summary = f"""
# Single-Cell Analysis Summary

## Dataset
- Cells: {adata.n_obs:,}
- Genes: {adata.n_vars:,}

## QC Statistics
- Median genes/cell: {adata.obs['n_genes_by_counts'].median():.0f}
- Median counts/cell: {adata.obs['total_counts'].median():.0f}
- Median mito%: {adata.obs['pct_counts_mt'].median():.1f}%

## Clustering
- Clusters: {adata.obs[cluster_key].nunique() if cluster_key else 'N/A'}
"""

print(summary)
put(summary, 'sc_summary.md')

In [ ]:
print("\n=== Analysis Complete ===")
print("Generated outputs:")
print("  - qc_metrics.png")
print("  - qc_scatter.png")
print("  - umap_clusters.png")
print("  - marker_expression.png")
print("  - sc_summary.md")